# Framework SCM – Motor de Velos
## Environmental Modulation in Galaxy Outer Rotation Slopes

**Autor:** Sergio Cámara Madrid  
**DOI:** [10.5281/zenodo.19455777](https://doi.org/10.5281/zenodo.19455777)  
**Repositorio:** [Motor-de-Velos-SCM](https://github.com/sergiocamaramadrid-cyber/Motor-de-Velos-SCM)

---

Este notebook ejecuta el análisis completo del Framework SCM en un solo clic.

Pasos que ejecuta:
1. Carga datos desde el repositorio
2. Calcula ΔF3 (pendiente externa)
3. Mass threshold scan
4. Correlación en alta masa
5. Análisis de residuos
6. Tests de robustez (Bootstrap, Permutación, Control de outliers)
7. Genera figuras automáticamente
8. Emite veredicto final

In [ ]:
# ── 0. Setup – instalar dependencias si estamos en Colab ──────────────────
import sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'numpy', 'pandas', 'scipy', 'statsmodels', 'matplotlib'],
                   check=True)

import io, textwrap, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy import stats
import statsmodels.formula.api as smf

warnings.filterwarnings('ignore')
np.random.seed(42)

print('Dependencias cargadas correctamente ✓')

In [ ]:
# ── 1. Cargar datos ───────────────────────────────────────────────────────
DATA_URL = ('https://raw.githubusercontent.com/sergiocamaramadrid-cyber/'
            'Motor-de-Velos-SCM/main/results/scm_master_final.csv')

try:
    df = pd.read_csv(DATA_URL)
    print(f'Datos cargados desde GitHub: {len(df)} galaxias')
except Exception:
    # Fallback: ruta local (si se ejecuta localmente)
    import pathlib
    local = pathlib.Path('../results/scm_master_final.csv')
    df = pd.read_csv(local)
    print(f'Datos cargados desde ruta local: {len(df)} galaxias')

# Columnas requeridas
REQUIRED = ['logMbar', 'slope_tail', 'env_proxy']
missing = [c for c in REQUIRED if c not in df.columns]
assert not missing, f'Columnas faltantes: {missing}'

df.head()

In [ ]:
# ── 2. Calcular ΔF3 (pendiente externa normalizada) ───────────────────────
# ΔF3 = slope_tail relativo a la media de baja masa
LOW_MASS_THRESHOLD = 10.0
low_mass_mask = df['logMbar'] < LOW_MASS_THRESHOLD
reference_slope = df.loc[low_mass_mask, 'slope_tail'].mean()

df['delta_f3_computed'] = df['slope_tail'] - reference_slope

print(f'Pendiente de referencia (logM < {LOW_MASS_THRESHOLD}): {reference_slope:.4f}')
print(f'ΔF3 medio (total): {df["delta_f3_computed"].mean():.4f}')
print(f'ΔF3 std:           {df["delta_f3_computed"].std():.4f}')

In [ ]:
# ── 3. Mass Threshold Scan ────────────────────────────────────────────────
scan_results = []
thresholds = np.arange(9.5, 11.3, 0.1)

for thr in thresholds:
    subset = df[df['logMbar'] >= thr]
    n = len(subset)
    if n < 5:
        continue
    rho, p = stats.spearmanr(subset['env_proxy'], subset['slope_tail'])
    scan_results.append({'threshold': round(thr, 2), 'n': n, 'rho': rho, 'p_value': p,
                         'log10_p': np.log10(p) if p > 0 else -30})

scan_df = pd.DataFrame(scan_results)
print(scan_df.to_string(index=False))

In [ ]:
# ── 4. Correlación en alta masa ───────────────────────────────────────────
HIGH_MASS_THRESHOLD = 10.6
high_mass = df[df['logMbar'] >= HIGH_MASS_THRESHOLD].copy()

rho_hm, p_hm = stats.spearmanr(high_mass['env_proxy'], high_mass['slope_tail'])
r_hm, p_pearson = stats.pearsonr(high_mass['env_proxy'], high_mass['slope_tail'])

print(f'Alta masa (logM ≥ {HIGH_MASS_THRESHOLD}): N = {len(high_mass)}')
print(f'  Spearman ρ = {rho_hm:.4f},  p = {p_hm:.2e}')
print(f'  Pearson  r = {r_hm:.4f},   p = {p_pearson:.2e}')

In [ ]:
# ── 5. Análisis de residuos (OLS con control de masa) ─────────────────────
model = smf.ols('slope_tail ~ logMbar + env_proxy', data=high_mass).fit(cov_type='HC3')
high_mass['residuals'] = model.resid

rho_res, p_res = stats.spearmanr(high_mass['env_proxy'], high_mass['residuals'])

print('OLS Alta Masa (HC3):')
print(model.summary().tables[1])
print(f'\nCorrelación residuos ~ env_proxy: ρ = {rho_res:.4f}, p = {p_res:.2e}')

In [ ]:
# ── 6a. Robustez: Bootstrap ───────────────────────────────────────────────
N_BOOT = 1000
boot_rhos = []
rng = np.random.default_rng(42)

env_vals = high_mass['env_proxy'].values
slope_vals = high_mass['slope_tail'].values

for _ in range(N_BOOT):
    idx = rng.integers(0, len(high_mass), size=len(high_mass))
    rho_b, _ = stats.spearmanr(env_vals[idx], slope_vals[idx])
    boot_rhos.append(rho_b)

boot_rhos = np.array(boot_rhos)
ci_low, ci_high = np.percentile(boot_rhos, [2.5, 97.5])

print(f'Bootstrap ({N_BOOT} iteraciones):')
print(f'  ρ observado: {rho_hm:.4f}')
print(f'  IC 95%: [{ci_low:.4f}, {ci_high:.4f}]')
print(f'  Fracción ρ < 0: {(boot_rhos < 0).mean():.3f}')

In [ ]:
# ── 6b. Robustez: Test de Permutación ────────────────────────────────────
N_PERM = 5000
perm_rhos = []

for _ in range(N_PERM):
    perm_slope = rng.permutation(slope_vals)
    rho_p, _ = stats.spearmanr(env_vals, perm_slope)
    perm_rhos.append(rho_p)

perm_rhos = np.array(perm_rhos)
p_perm = (perm_rhos <= rho_hm).mean()  # fracción tan extrema o más

print(f'Test de permutación ({N_PERM} iteraciones):')
print(f'  ρ observado: {rho_hm:.4f}')
print(f'  p-valor permutación: {p_perm:.4f}')

In [ ]:
# ── 6c. Robustez: Control de outliers (eliminación jackknife) ─────────────
jack_rhos = []
jack_ids = []

for i in range(len(high_mass)):
    mask = np.ones(len(high_mass), dtype=bool)
    mask[i] = False
    rho_j, _ = stats.spearmanr(env_vals[mask], slope_vals[mask])
    jack_rhos.append(rho_j)
    jack_ids.append(i)

jack_rhos = np.array(jack_rhos)
print(f'Jackknife (leave-one-out):')
print(f'  ρ mín: {jack_rhos.min():.4f}   ρ máx: {jack_rhos.max():.4f}')
print(f'  Todos negativos: {(jack_rhos < 0).all()}')

In [ ]:
# ── 7. Figuras ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Framework SCM – Motor de Velos', fontsize=14, fontweight='bold')

# Panel 1: Mass threshold scan
ax = axes[0]
ax.plot(scan_df['threshold'], scan_df['rho'], 'o-', color='steelblue', lw=2)
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.axvline(HIGH_MASS_THRESHOLD, color='red', lw=1.2, ls=':', label=f'logM={HIGH_MASS_THRESHOLD}')
ax.set_xlabel('logM threshold', fontsize=11)
ax.set_ylabel('Spearman ρ (env_proxy vs slope_tail)', fontsize=10)
ax.set_title('Mass Threshold Scan', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Panel 2: Alta masa – dispersión env vs slope
ax = axes[1]
sc = ax.scatter(high_mass['env_proxy'], high_mass['slope_tail'],
                c=high_mass['logMbar'], cmap='plasma', s=60, edgecolors='k', lw=0.4)
m, b = np.polyfit(high_mass['env_proxy'], high_mass['slope_tail'], 1)
xr = np.linspace(high_mass['env_proxy'].min(), high_mass['env_proxy'].max(), 100)
ax.plot(xr, m*xr+b, 'r--', lw=1.5, label=f'OLS (ρ={rho_hm:.2f}, p={p_hm:.2e})')
plt.colorbar(sc, ax=ax, label='logMbar')
ax.set_xlabel('env_proxy (Mpc a grupo)', fontsize=11)
ax.set_ylabel('slope_tail (pendiente exterior)', fontsize=10)
ax.set_title(f'Alta Masa (logM ≥ {HIGH_MASS_THRESHOLD})', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Panel 3: Bootstrap distribution
ax = axes[2]
ax.hist(boot_rhos, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(rho_hm, color='red', lw=2, label=f'ρ obs = {rho_hm:.3f}')
ax.axvline(ci_low, color='orange', lw=1.5, ls='--', label=f'IC 95% [{ci_low:.3f}, {ci_high:.3f}]')
ax.axvline(ci_high, color='orange', lw=1.5, ls='--')
ax.axvline(0, color='gray', lw=0.8)
ax.set_xlabel('Bootstrap ρ', fontsize=11)
ax.set_ylabel('Frecuencia', fontsize=11)
ax.set_title(f'Bootstrap (N={N_BOOT})', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('scm_framework_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada: scm_framework_results.png')

In [ ]:
# ── 8. Veredicto final ────────────────────────────────────────────────────
ALPHA = 0.05
signal_detected = (p_hm < ALPHA) and (rho_hm < 0) and (ci_high < 0) and (p_perm < ALPHA)

verdict = '✅ SEÑAL AMBIENTAL DETECTADA' if signal_detected else '⚠️  SEÑAL NO SIGNIFICATIVA'

print('=' * 60)
print('     VEREDICTO FINAL – Framework SCM Motor de Velos')
print('=' * 60)
print(f'  N galaxias (total):        {len(df)}')
print(f'  N alta masa (logM≥{HIGH_MASS_THRESHOLD}):   {len(high_mass)}')
print(f'  Spearman ρ:                {rho_hm:.4f}')
print(f'  p-valor (Spearman):        {p_hm:.2e}')
print(f'  IC 95% Bootstrap:          [{ci_low:.4f}, {ci_high:.4f}]')
print(f'  p-valor (permutación):     {p_perm:.4f}')
print(f'  Jackknife ρ siempre <0:    {(jack_rhos < 0).all()}')
print('-' * 60)
print(f'  {verdict}')
print('=' * 60)
print()
print('Interpretación:')
if signal_detected:
    print('  Las galaxias de alta masa en entornos más densos muestran')
    print('  pendientes externas sistemáticamente más negativas.')
    print('  La señal es robusta frente a bootstrap, permutación')
    print('  y eliminación de outliers (jackknife).')
else:
    print('  No se detecta señal significativa con los datos actuales.')